# Stage 09: Feature Engineering

**Project context:** SPY Next-Day High-Volatility Risk Alert. Features are built from data available after each close; the next-day outcome is retained only for diagnostic checks.


## 1. Reproducible setup and data


In [1]:
from pathlib import Path
import os, sys
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
import pandas as pd
from IPython.display import display

if Path.cwd().name == "homework09":
    ROOT = Path.cwd()
else:
    ROOT = Path.cwd() / "homework" / "homework09"
if not (ROOT / "src" / "features.py").is_file():
    raise FileNotFoundError("Run from repository root or homework09.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.features import build_spy_features, feature_target_correlation
RAW, PROCESSED, REPORTS = ROOT / "data/raw", ROOT / "data/processed", ROOT / "reports"
REPORTS.mkdir(parents=True, exist_ok=True)
print("Homework root:", ROOT)


Homework root: /Users/cengchengyu/Documents/NYU/Class/Boot Camp 4/CS HW/bootcamp_chengyu_zeng/homework/homework09


## 2. Feature construction and time alignment

`return_t`, range, volume change, rolling volatility, and weekday encoding use information through date `t`. `next_day_abs_return` is shifted backward as an outcome only.


In [2]:
raw_path = RAW / "spy_ohlcv_stage08_snapshot.parquet"
raw = pd.read_parquet(raw_path)
features = build_spy_features(raw)
correlations = feature_target_correlation(features)
feature_path = PROCESSED / "spy_feature_candidates.csv"
correlation_path = PROCESSED / "feature_target_correlations.csv"
features.to_csv(feature_path, index=False, date_format="%Y-%m-%d")
correlations.to_csv(correlation_path, index=False)
print("Raw shape:", raw.shape)
print("Feature shape:", features.shape)
display(features.head())
display(correlations)


Raw shape: (2512, 6)
Feature shape: (2512, 18)


,date,open,high,low,close,volume,return_t,abs_return_t,intraday_range_t,log_volume_change_t,rolling_volatility_5_t,stress_interaction_t,weekday_Friday,weekday_Monday,weekday_Thursday,weekday_Tuesday,weekday_Wednesday,next_day_abs_return
0,2016-08-23,219.25,219.600,218.90,218.9700,53289030,NaN,NaN,0.003193,NaN,NaN,NaN,0,0,0,1,0,0.005115
1,2016-08-24,218.80,218.910,217.36,217.8500,71553010,-0.005115,0.005115,0.007084,0.294708,NaN,0.000036,0,0,0,0,1,0.000689
2,2016-08-25,217.40,218.190,217.22,217.7000,69128600,-0.000689,0.000689,0.004462,-0.034470,NaN,0.000003,0,0,1,0,0,0.001883
3,2016-08-26,217.92,219.120,216.25,217.2900,122386400,-0.001883,0.001883,0.013170,0.571215,NaN,0.000025,1,0,0,0,0,0.004928
4,2016-08-29,217.44,218.665,217.40,218.3608,70431280,0.004928,0.004928,0.005818,-0.552546,NaN,0.000029,0,1,0,0,0,0.001652


,feature,observations,pearson_correlation
0,intraday_range_t,2511,0.506382
1,rolling_volatility_5_t,2506,0.497759
2,stress_interaction_t,2510,0.403518
3,abs_return_t,2510,0.358598
4,return_t,2510,-0.109682
5,weekday_Thursday,2511,0.029409
6,weekday_Monday,2511,-0.021626
7,log_volume_change_t,2510,0.011774
8,weekday_Tuesday,2511,-0.010845
9,weekday_Wednesday,2511,0.009132


## 3. Correlation diagnostic

Correlation is a screening tool. It does not prove that a feature predicts the future or improves a later chronological model.


In [3]:
plot_path = REPORTS / "feature_target_correlations.png"
plot_data = correlations.sort_values("pearson_correlation")
fig, ax = plt.subplots(figsize=(9, 5.5))
colors = ["#DC2626" if value < 0 else "#2563EB" for value in plot_data["pearson_correlation"]]
ax.barh(plot_data["feature"], plot_data["pearson_correlation"], color=colors)
ax.axvline(0, color="#374151", linewidth=0.8)
ax.set_title("Pairwise Correlation with Next-Day Absolute Return")
ax.set_xlabel("Pearson correlation")
fig.tight_layout()
fig.savefig(plot_path, dpi=170, bbox_inches="tight")
plt.close(fig)
print("Saved:", plot_path.name)


Saved: feature_target_correlations.png


## 4. Interpretation and risks

- The engineered features follow Stage08 findings about tail risk, range, volume, and volatility clustering.
- Weekday is one-hot encoded because it is categorical and unordered; it should not be label encoded as if weekdays have numeric rank.
- Feature correlations are not causal evidence. Stage10 must use chronological splits and assess incremental out-of-sample value.
- Do not impute the feature warm-up rows or terminal next-day outcome with future-aware values.


In [4]:
assert raw.shape == (2512, 6)
assert len(features) == len(raw)
assert int(features["return_t"].isna().sum()) == 1
assert int(features["rolling_volatility_5_t"].isna().sum()) == 5
assert int(features["next_day_abs_return"].isna().sum()) == 1
assert {"weekday_Monday", "weekday_Tuesday", "weekday_Wednesday", "weekday_Thursday", "weekday_Friday"}.issubset(features.columns)
assert feature_path.exists() and correlation_path.exists() and plot_path.exists()
print("Stage09 feature and artifact checks passed.")


Stage09 feature and artifact checks passed.
